In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- query_appraised ---
FIX_QUERY_APPRAISED_PIPE_READ_PD = pd.DataFrame({"gene": ["g1", "g2", "g3"], "sample": ["s1", "s2", "s3"], "sequence": ["seq1", "seq2", "seq3"], "num_hits": [1, 2, 1], "coverage": [0.9, 0.8, 0.95], "taxonomy": ["Bacteria", "Archaea", "Bacteria"], "divergence": [0.01, 0.02, 0.01]})
FIX_QUERY_APPRAISED_QUERY_READ_PD = pd.DataFrame({"marker": ["g1", "g2", "g3"], "query_name": ["s1", "s2", "s3"], "query_sequence": ["seq1", "seq2", "seq3"], "sample": ["s2", "s3", "s1"], "num_hits": [1, 2, 1], "coverage": [0.9, 0.8, 0.95], "taxonomy": ["Bacteria", "Archaea", "Bacteria"]})
FIX_QUERY_APPRAISED_PIPE_READ_PL = pl.from_pandas(FIX_QUERY_APPRAISED_PIPE_READ_PD)
FIX_QUERY_APPRAISED_QUERY_READ_PL = pl.from_pandas(FIX_QUERY_APPRAISED_QUERY_READ_PD)
FIX_QUERY_APPRAISED_PIPE_READ = FIX_QUERY_APPRAISED_PIPE_READ_PD
FIX_QUERY_APPRAISED_QUERY_READ = FIX_QUERY_APPRAISED_QUERY_READ_PD

# --- query_binned_unbinned ---
FIX_QUERY_OUTPUT_COLUMNS = ["gene", "sample", "sequence", "num_hits", "coverage", "taxonomy", "found_in"]
FIX_QUERY_APPRAISED_BINNED_PD = pd.DataFrame({
    "gene": ["g1", "g2", "g3"], "sample": ["s1", "s2", "s3"], "sequence": ["seq1", "seq2", "seq3"],
    "num_hits": [1, 2, 1], "coverage": [0.9, 0.8, 0.95], "taxonomy": ["Bacteria", "Archaea", "Bacteria"],
    "found_in": ["s2", "s3", "s1"], "divergence": [0.01, 0.02, 0.03], "binned": [True, False, False],
})
FIX_QUERY_PIPE_READ_PD = pd.DataFrame({
    "gene": ["g1", "g2", "g3"], "sample": ["s1", "s2", "s3"], "sequence": ["seq1", "seq2", "seq3"],
    "num_hits": [1, 2, 1], "coverage": [0.9, 0.8, 0.95], "taxonomy": ["Bacteria", "Archaea", "Bacteria"],
})
FIX_QUERY_APPRAISED_BINNED_PL = pl.from_pandas(FIX_QUERY_APPRAISED_BINNED_PD)
FIX_QUERY_PIPE_READ_PL = pl.from_pandas(FIX_QUERY_PIPE_READ_PD)

# --- query_write_csv ---
FIX_QUERY_WRITE_CSV_OUTPUTS_PD = [(pd.DataFrame({"id": [1, 2], "value": [10, 20]}), pd.DataFrame({"id": [3], "value": [30]}))]
FIX_QUERY_WRITE_CSV_OUTPUTS_PL = [(pl.from_pandas(b), pl.from_pandas(u)) for b, u in FIX_QUERY_WRITE_CSV_OUTPUTS_PD]
FIX_QUERY_WRITE_CSV_OUTPUTS = FIX_QUERY_WRITE_CSV_OUTPUTS_PD
FIX_QUERY_WRITE_CSV_BINNED_PATH = "test_file_binned.tsv"
FIX_QUERY_WRITE_CSV_UNBINNED_PATH = "test_file_unbinned.tsv"

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_query_appraised(pipe_read, query_read):
    appraised = (query_read
        .rename(columns = {"marker": "gene", "query_name":"sample", "query_sequence": "sequence", "sample": "found_in"})
        .drop(["taxonomy", "num_hits", "coverage"], axis=1, errors="ignore")
        .set_index(["gene", "sample", "sequence"])
        .join(pipe_read.set_index(["gene", "sample", "sequence"]), on = ["gene", "sample", "sequence"], how = "inner")
        .reset_index()
        .groupby(["gene", "sample", "sequence", "num_hits", "coverage", "taxonomy", "divergence"])["found_in"]
        .agg(lambda x: ",".join(sorted(x)))
        .reset_index()
        )
    return appraised

def before_query_binned_unbinned():
    def before_binned_unbinned(
        appraised: pd.DataFrame,
        pipe_read: pd.DataFrame,
        output_columns: list,
    ) -> tuple:
        binned = (appraised[appraised["binned"]]
                  .drop(["divergence","binned"], axis=1)
                  .reset_index(drop=True))
        unbinned = pipe_read.join(
            appraised.set_index(output_columns[0:-1]),
            on=output_columns[0:-1],
        )
        unbinned = (unbinned[~unbinned["binned"].fillna(False)]
                    .drop(["divergence","binned"], axis=1)
                    .reset_index(drop=True))
        unbinned["found_in"] = None
        return binned, unbinned
    return before_binned_unbinned

def before_query_write_csv(binned_path, outputs, unbinned_path):
    first = True
    for binned, unbinned in outputs:
        binned.to_csv(binned_path, sep = "\t", mode = "a", header = first, index = False)
        unbinned.to_csv(unbinned_path, sep = "\t", mode = "a", header = first, index = False)
        first = False
    return first

In [ ]:
# ── Generated wrappers (experiment-generated Polars) ─────────────────────────

def gen_query_appraised(pipe_read, query_read):

    appraised = (
        query_read
        .rename({"marker": "gene", "query_name": "sample", "query_sequence": "sequence", "sample": "found_in"})
        .drop(["taxonomy", "num_hits", "coverage"], strict=False)
        .join(pipe_read, on=["gene", "sample", "sequence"], how="inner")
        .group_by(["gene", "sample", "sequence", "num_hits", "coverage", "taxonomy", "divergence"])
        .agg(pl.col("found_in").sort().list.join(",").alias("found_in"))
        .sort(["gene", "sample", "sequence", "num_hits", "coverage", "taxonomy", "divergence"])
    )
    return appraised

def gen_query_binned_unbinned():
    def before_binned_unbinned(
        appraised: pl.DataFrame,
        pipe_read: pl.DataFrame,
        output_columns: list,
    ) -> tuple:
        binned = (
            appraised.filter(pl.col("binned"))
            .drop(["divergence", "binned"])
        )
        unbinned = pipe_read.join(
            appraised,
            on=output_columns[0:-1],
            how="left",
        )
        unbinned = (
            unbinned.filter(~pl.col("binned").fill_null(False))
            .drop(["divergence", "binned"])
        )
        unbinned = unbinned.with_columns(pl.lit(None).alias("found_in"))
        return binned, unbinned
    return before_binned_unbinned

def gen_query_write_csv(binned_path, outputs, unbinned_path):
    first = True
    for binned, unbinned in outputs:
        with open(binned_path, "a") as f:
            binned.write_csv(f, separator="\t", include_header=first)
        with open(unbinned_path, "a") as f:
            unbinned.write_csv(f, separator="\t", include_header=first)
        first = False
    return first

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: query_appraised ===

try:
    _r = gen_query_appraised(FIX_QUERY_APPRAISED_PIPE_READ_PL, FIX_QUERY_APPRAISED_QUERY_READ_PL)
    print("✅ L1 smoke gen_query_appraised: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_query_appraised: {type(_e).__name__}: {_e}")

try:
    _rb = before_query_appraised(FIX_QUERY_APPRAISED_PIPE_READ_PD, FIX_QUERY_APPRAISED_QUERY_READ_PD)
    print("✅ L1 smoke before_query_appraised: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_query_appraised: {type(_e).__name__}: {_e}")

try:
    _rb = before_query_appraised(FIX_QUERY_APPRAISED_PIPE_READ_PD, FIX_QUERY_APPRAISED_QUERY_READ_PD)
    _rg = gen_query_appraised(FIX_QUERY_APPRAISED_PIPE_READ_PL, FIX_QUERY_APPRAISED_QUERY_READ_PL)
    compare(_rb, _rg, "query_appraised")
except Exception as _e:
    print(f"❌ L2 equivalence query_appraised: setup error — {type(_e).__name__}: {_e}")

try:
    _rb = before_query_appraised(FIX_QUERY_APPRAISED_PIPE_READ_PD.head(0), FIX_QUERY_APPRAISED_QUERY_READ_PD.head(0))
    _rg = gen_query_appraised(FIX_QUERY_APPRAISED_PIPE_READ_PL.head(0), FIX_QUERY_APPRAISED_QUERY_READ_PL.head(0))
    compare(_rb, _rg, "L3 edge query_appraised empty")
except Exception as _e:
    print(f"❌ L3 edge query_appraised empty: {type(_e).__name__}: {_e}")
